# 005 · Serialisation with pickle

> **Standard Library › Data Formats** · Python 3.14

`pickle` turns (almost) any Python object graph into bytes and back again — your own classes, nested containers, shared references, cycles — with zero configuration. That power comes with a serious price: **loading a pickle can execute arbitrary code**.

**In this notebook you will learn:**

- how `pickle` relates to .NET's `BinaryFormatter` — and why both are dangerous
- `dumps`/`loads` (bytes) and `dump`/`load` (files)
- what can and cannot be pickled (functions by reference, no lambdas)
- protocols, and customising with `__getstate__` / `__setstate__`
- how `copy.deepcopy` uses the same machinery
- when to use JSON instead (most of the time, for interchange)

## .NET mental model

| .NET | Python |
|---|---|
| `BinaryFormatter.Serialize(stream, obj)` | `pickle.dump(obj, f)` |
| `BinaryFormatter.Deserialize(stream)` | `pickle.load(f)` |
| `[Serializable]` attribute | nothing — almost everything is picklable by default |
| `[NonSerialized]` field | exclude it in `__getstate__` |
| `ISerializable` / `[OnDeserialized]` | `__getstate__` / `__setstate__` (or `__reduce__`) |
| `SerializationException` | `pickle.PicklingError` / `pickle.UnpicklingError` / `TypeError` |

`BinaryFormatter` was marked obsolete in .NET 5 (warning `SYSLIB0011`) and its implementation was **removed in .NET 9**, because deserialising untrusted data with it allows remote code execution. Microsoft's guidance is that it "can't be made secure".

`pickle` has **exactly the same problem** — it is still in the standard library because it is very useful for *trusted* data (caches, inter-process communication, saving your own objects), but the rule is absolute:

> ⚠️ **Never unpickle data from an untrusted or unauthenticated source.**

## Bytes: `dumps` and `loads`

The result is `bytes`, not text. Shared references and cycles survive the round trip — something JSON cannot do.

In [ ]:
import pickle
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal

@dataclass
class Order:
    id: int
    placed: datetime
    total: Decimal
    tags: set[str]

order = Order(1001, datetime(2025, 3, 1, 12, 30), Decimal("19.99"), {"gift"})

data = pickle.dumps(order)
print(type(data).__name__, len(data), data[:16])

copy_of_order = pickle.loads(data)
print(copy_of_order)
print(copy_of_order == order, copy_of_order is order)   # equal, but a new object

Notice that `datetime`, `Decimal` and `set` needed no hooks at all — compare that with the `default=` function required by `json`.

In [ ]:
node = {"name": "root", "children": []}
node["children"].append(node)                 # a cycle
shared = [1, 2]
pair = (shared, shared)                       # the same list twice

node2 = pickle.loads(pickle.dumps(node))
print(node2["children"][0] is node2)          # True — the cycle is preserved

pair2 = pickle.loads(pickle.dumps(pair))
print(pair2[0] is pair2[1])                   # True — sharing is preserved

## Files: `dump` and `load`

Files must be opened in **binary** mode (`"wb"` / `"rb"`). You can write several objects to one file and read them back in order.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.pkl"

    with path.open("wb") as f:
        pickle.dump(order, f)
        pickle.dump({"count": 1}, f)

    with path.open("rb") as f:
        first = pickle.load(f)
        second = pickle.load(f)

print(first.id, second)

## ⚠️ Why loading untrusted pickles is dangerous

A pickle is not data — it is a small **program** for a stack-based virtual machine. An object can tell pickle "to recreate me, call this function with these arguments" via `__reduce__`. A malicious file can name *any* importable callable (`os.system`, `subprocess.run`, ...).

Here is a harmless demonstration that uses `print` instead:

In [ ]:
class Innocent:
    def __reduce__(self):
        return (print, ("⚠️ this code ran during pickle.loads()!",))

payload = pickle.dumps(Innocent())
result = pickle.loads(payload)    # executes print(...) — it could have been anything
print(result)                     # the "object" is just print's return value

This is the same class of vulnerability that killed `BinaryFormatter`. Mitigations, in order of preference:

1. Don't accept pickles from outside your trust boundary — use JSON (or another data-only format).
2. If you must move pickles between your own services, sign them with `hmac` and verify **before** loading *(see Standard Library › Data Formats › 006 · Encoding and Hashing with base64 and hashlib)*.
3. A restricted `Unpickler.find_class` allow-list reduces the risk but is easy to get wrong.

## What can be pickled?

- ✅ `None`, booleans, numbers, strings, bytes, and containers of picklable objects
- ✅ instances of classes defined at module top level (their `__dict__` is pickled)
- ✅ functions and classes defined at module top level — **by reference** (module + name only, not their code)
- ❌ lambdas, nested functions, and local classes (no importable name)
- ❌ live resources: open files, locks, sockets, database connections, generators

In [ ]:
import threading

def shout(text):
    return text.upper()

print(pickle.dumps(shout))                      # just a reference: module + name
print(pickle.loads(pickle.dumps(shout)) is shout)

candidates = {
    "lambda": lambda x: x,
    "lock": threading.Lock(),
    "generator": (n for n in range(3)),
}
for label, obj in candidates.items():
    try:
        pickle.dumps(obj)
    except (pickle.PicklingError, TypeError) as e:
        print(f"{label:9} -> {type(e).__name__}")

⚠️ **Gotcha:** because classes and functions are stored by reference, the **loading** program must be able to import the same module and name. Renaming or moving a class breaks old pickles — another reason pickle is a poor long-term storage format. (Objects defined in a notebook live in the `__main__` module, so their pickles only load back in a session that defines them too.)

## Protocols

The pickle format has evolved through several protocol versions. Newer protocols are more compact and faster; older ones are readable by older Pythons.

| Protocol | Notes |
|---|---|
| 0 | the original human-readable ASCII format |
| 2 | introduced in Python 2.3 — efficient pickling of new-style classes |
| 4 | Python 3.4+ — very large objects, more kinds of objects |
| 5 | Python 3.8+ — out-of-band buffers (PEP 574); the default since 3.14 |

In [ ]:
print(pickle.DEFAULT_PROTOCOL, pickle.HIGHEST_PROTOCOL)

sample = {"ids": list(range(10)), "name": "sensor-1"}
for protocol in range(pickle.HIGHEST_PROTOCOL + 1):
    print(protocol, len(pickle.dumps(sample, protocol=protocol)))

print(pickle.dumps(42, protocol=0))    # protocol 0 is ASCII

**Tip:** you rarely need to set the protocol. Pass `protocol=pickle.HIGHEST_PROTOCOL` when speed/size matters and only the same Python version will read the data.

## Customising with `__getstate__` and `__setstate__`

By default, pickle saves the instance `__dict__`. Override `__getstate__` to choose what is saved (like `[NonSerialized]`), and `__setstate__` to rebuild what wasn't (like `[OnDeserialized]`).

```csharp
// C# — the old BinaryFormatter way
[Serializable]
class PriceCache
{
    public string Currency;
    [NonSerialized] private object _lock = new();
    [OnDeserialized] void Restore(StreamingContext _) => _lock = new object();
}
```

In [ ]:
class PriceCache:
    def __init__(self, currency):
        self.currency = currency
        self.prices = {}
        self._lock = threading.Lock()          # not picklable!

    def __getstate__(self):
        state = self.__dict__.copy()
        del state["_lock"]                     # [NonSerialized]
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._lock = threading.Lock()          # [OnDeserialized]

cache = PriceCache("EUR")
cache.prices["apple"] = Decimal("0.50")

restored = pickle.loads(pickle.dumps(cache))
print(restored.currency, restored.prices, type(restored._lock).__name__)

Note that `__init__` is **not** called when unpickling — the object is created empty and its state is filled in. Validation you put in `__init__` will not run.

## Relation to `copy.deepcopy`

`copy.copy` and `copy.deepcopy` reuse the pickle protocol (`__reduce_ex__`, `__getstate__`, `__setstate__`) — they just skip the bytes. So the customisation above also makes `PriceCache` deep-copyable:

In [ ]:
import copy

clone = copy.deepcopy(cache)
print(clone.prices == cache.prices, clone.prices is cache.prices)
print(clone._lock is cache._lock)     # a fresh lock from __setstate__

try:
    copy.deepcopy(threading.Lock())   # same restriction as pickling
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

**Tip:** `pickle.loads(pickle.dumps(x))` is an old idiom for a deep copy — `copy.deepcopy(x)` is clearer and faster.

## Errors

| Error | When |
|---|---|
| `pickle.PicklingError` / `TypeError` | an object cannot be pickled |
| `pickle.UnpicklingError` | corrupt or truncated data (other errors such as `EOFError`, `AttributeError`, `ImportError` are possible too) |

In [ ]:
try:
    pickle.loads(b"not a pickle")
except pickle.UnpicklingError as e:
    print(f"{type(e).__name__}: {e}")

try:
    pickle.loads(pickle.dumps(order)[:-5])        # truncated
except pickle.UnpicklingError as e:
    print(f"{type(e).__name__}: {e}")

## Pickle or JSON?

| | `pickle` | `json` |
|---|---|---|
| Safe with untrusted input | ❌ never | ✅ yes (data only) |
| Readable by other languages | ❌ Python only | ✅ everywhere |
| Stable across refactors | ❌ breaks if classes move/rename | ✅ you control the schema |
| Custom classes, cycles, `datetime`, `set` | ✅ automatic | ❌ needs hooks |
| Typical use | caches, `multiprocessing`, short-lived trusted data | APIs, config, files, messages |

**Rule of thumb:** use JSON (or CSV, TOML, SQLite) for anything that crosses a process, team, or version boundary. Reach for pickle for trusted, short-lived, Python-to-Python data — which is exactly how `multiprocessing` and `shelve` use it.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Object → bytes | `BinaryFormatter.Serialize` (removed in .NET 9) | `pickle.dumps(obj)` |
| Bytes → object | `BinaryFormatter.Deserialize` | `pickle.loads(data)` |
| To / from a file | `Serialize(stream, obj)` | `pickle.dump(obj, f)` / `pickle.load(f)` (binary mode) |
| Exclude a field | `[NonSerialized]` | `__getstate__` |
| Restore after load | `[OnDeserialized]` | `__setstate__` |
| Deep copy | manual / serialise round trip | `copy.deepcopy(obj)` |
| Untrusted input | use `System.Text.Json` | use `json` — never `pickle` |

## Exercises

1. Pickle a list containing a `dict`, a `set` and a `datetime` to a file in a `TemporaryDirectory`, load it back and check that it equals the original.
2. Write a class `Connection` with attributes `host`, `port` and `_socket = object()` (pretend it's a live socket). Make pickling skip `_socket` and set it to `None` after unpickling.
3. Explain in one or two sentences why `pickle.loads(request_body)` in a web API is a security hole, and what you would use instead.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import pickle
import tempfile
from datetime import datetime
from pathlib import Path

# 1 — binary mode on both sides
data = [{"a": 1}, {"x", "y"}, datetime(2025, 1, 1, 9, 0)]
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.pkl"
    with path.open("wb") as f:
        pickle.dump(data, f)
    with path.open("rb") as f:
        print(pickle.load(f) == data)   # True

# 2 — drop the live resource in __getstate__, recreate a placeholder in __setstate__
class Connection:
    def __init__(self, host, port):
        self.host, self.port = host, port
        self._socket = object()

    def __getstate__(self):
        return {k: v for k, v in self.__dict__.items() if k != "_socket"}

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._socket = None

c = pickle.loads(pickle.dumps(Connection("localhost", 5432)))
print(c.host, c.port, c._socket)   # localhost 5432 None

# 3 — unpickling runs code chosen by whoever produced the bytes (via __reduce__),
#     so an attacker controlling request_body gets remote code execution.
#     Use json.loads (data only) and validate the result instead.
```

</details>